In [1]:
import torch
import torch.nn as nn

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")


Using cpu device


In [5]:
class RegularizedRNN(nn.Module):
    def __init__(self, n_inputs, n_neurons, n_outputs, tau_ms=50.0, dt_ms=5.0, g=1.2):
        """
        Sussillo et al. (2015)의 연속 시간 RNN 모델 초기화.

        Args:
            n_inputs (int): 입력 차원 (I)
            n_neurons (int): 순환 유닛(뉴런)의 수 (N)
            n_outputs (int): 출력 차원 (M)
            tau_ms (float): 뉴런의 시간 상수 (τ)
            dt_ms (float): 시뮬레이션 시간 스텝 (Δt)
            g (float): 순환 가중치 행렬의 스케일 (g)
        """
        super().__init__()
        self.n_inputs = n_inputs
        self.n_neurons = n_neurons
        self.n_outputs = n_outputs
        
        # 시간 관련 파라미터
        self.tau = tau_ms
        self.dt = dt_ms
        self.alpha_dt = self.dt / self.tau  # 오일러 적분을 위한 상수

        # 입력 가중치 행렬 B 초기화 (I -> N)
        self.B = nn.Linear(n_inputs, n_neurons, bias=False)
        nn.init.xavier_normal_(self.B.weight)

        # 순환 가중치 행렬 J 초기화 (N -> N)
        self.J = nn.Linear(n_neurons, n_neurons)
        # 초기 가중치를 g/sqrt(N) 스케일의 정규분포에서 샘플링
        initial_J = torch.randn(n_neurons, n_neurons) * (g / (n_neurons**0.5))
        self.J.weight.data = initial_J
        
        # 출력 가중치 행렬 W 초기화 (N -> M)
        self.W = nn.Linear(n_neurons, n_outputs)
        nn.init.xavier_normal_(self.W.weight)

        # 편향 벡터 b^x, b^z
        self.b_x = self.J.bias # J의 bias를 b_x로 사용
        nn.init.zeros_(self.b_x)
        self.b_z = self.W.bias # W의 bias를 b_z로 사용
        nn.init.zeros_(self.b_z)

    def forward(self, u, initial_x=None):
        """
        RNN의 순방향 패스를 시뮬레이션.

        Args:
            u (Tensor): 입력 시계열 데이터 (batch_size, n_timesteps, n_inputs)
            initial_x (Tensor, optional): 초기 은닉 상태. Defaults to None (zeros).

        Returns:
            Tuple: 출력 z와 은닉 상태 x의 시계열
        """
        batch_size, n_timesteps, _ = u.shape
        
        # 초기 은닉 상태 x 설정
        if initial_x is None:
            x = torch.zeros(batch_size, self.n_neurons, device=u.device)
        else:
            x = initial_x

        # 결과를 저장할 리스트
        x_history = []
        z_history = []

        # 오일러 적분을 사용하여 시간 스텝별로 시뮬레이션
        for t in range(n_timesteps):
            # 발화율 r 계산 (활성화 함수: tanh)
            r = torch.tanh(x)
            
            # 상태 업데이트 방정식: τ * dx/dt = -x + J*r + B*u + b_x
            # 오일러 적분: x_new = x + (dt/τ) * (-x + J*r + B*u + b_x)
            dx = -x + self.J(r) + self.B(u[:, t, :])
            x = x + self.alpha_dt * dx
            
            # 출력 z 계산
            z = self.W(r)
            
            # 히스토리 저장
            x_history.append(x)
            z_history.append(z)
            
        # 텐서로 변환
        x_out = torch.stack(x_history, dim=1)
        z_out = torch.stack(z_history, dim=1)
        
        return z_out, x_out

In [ ]:
import torch.optim as optim

def train_model(model, data_loader, n_epochs=500, lr=1e-3, 
                alpha_l2=1e-5, beta_fr=1e-3, gamma_j=1e-2):
    """
    정규화된 손실 함수를 사용하여 RNN 모델을 훈련.

    Args:
        model (nn.Module): 훈련할 RegularizedRNN 모델
        data_loader (DataLoader): 훈련 데이터 로더
        n_epochs (int): 훈련 에포크 수
        lr (float): 학습률
        alpha_l2, beta_fr, gamma_j (float): 정규화 하이퍼파라미터
    """
    # 최적화기 설정 (Adam)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    # 손실 함수 (MSE)
    mse_loss = nn.MSELoss()
    
    loss_history = []

    for epoch in range(n_epochs):
        epoch_loss = 0.0
        for u_batch, target_z_batch in data_loader:
            # 순방향 패스
            pred_z, x_states = model(u_batch)
            
            # 1. EMG 재현 오차 (E)
            loss_e = mse_loss(pred_z, target_z_batch)
            
            # 2. L2 가중치 정규화 (R_L2)
            loss_l2 = torch.sum(model.B.weight**2) + torch.sum(model.W.weight**2)
            
            # 3. 발화율 정규화 (R_FR)
            r_states = torch.tanh(x_states)
            loss_fr = torch.mean(r_states**2)
            
            # 4. 자코비안 정규화 (R_J) - 논문의 단순화된 그래디언트 근사
            # 실제 구현에서는 이 항의 그래디언트를 직접 계산하기보다,
            # 순환 가중치 J 자체에 대한 L2 페널티로 근사하는 것이 일반적이고 효과적입니다.
            # 이는 R_J가 J의 크기에 비례하여 증가하는 경향이 있기 때문입니다.
            loss_j_approx = torch.sum(model.J.weight**2)

            # 전체 정규화된 손실 함수
            total_loss = loss_e + alpha_l2 * loss_l2 + beta_fr * loss_fr + gamma_j * loss_j_approx
            
            # 역전파 및 최적화
            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()
            
            epoch_loss += total_loss.item()
        
        avg_epoch_loss = epoch_loss / len(data_loader)
        loss_history.append(avg_epoch_loss)
        if (epoch + 1) % 50 == 0:
            print(f'Epoch [{epoch+1}/{n_epochs}], Loss: {avg_epoch_loss:.4f}')
            
    return model, loss_history